# Train model on adult

In [8]:
import sys; sys.path.append("../")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, confusion_matrix, accuracy_score

from optimization_research.ml_constraints import BinarySolver

In [9]:
df = pd.read_csv("adult.csv")

In [10]:
#X and y
X = df.drop("income", axis=1)
y = df["income"].map({"<=50K": 0, ">50K": 1})

cat_cols = X.select_dtypes(include=["object"]).columns
X[cat_cols] = X[cat_cols].astype("category")

X_dummies = pd.get_dummies(X[cat_cols], drop_first=True)
X = pd.concat([X, X_dummies], axis=1)
X= X.drop(cat_cols, axis=1)

In [11]:
#preprocess data
# X = pd.get_dummies(X, drop_first=True)
# X = X.fillna(X.mean())

#Train and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#Train and validation split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

#XGBoost
model = xgb.XGBClassifier()
model.fit(X_train, y_train)

#Predictions
y_pred = model.predict(X_val)

#Accuracy
accuracy = accuracy_score(y_val, y_pred)

if accuracy>0.8:
    print("Model is good, we can move on to production")
    

Model is good, we can move on to production


### Using the model on the batch

In [12]:
N = len(X_test)
y_hat = model.predict_proba(X_test)[:, 1] #probability of class 1, shape (N,) 

groups = (X_test["age"]>30).map({True: "old", False: "young"}) #Each individual belongs to a group, shape (N,)

#Cost matrix
cost_matrix = np.array([[0, 3], 
                        [1, 0]]) 
cost_matrix = cost_matrix[None, :,:].repeat(N, axis=0) #broadcasting (N, 2, 2), could be configured for each individual

#The global constraint, the sum of the positive decisions should be less than 100
global_constraint = 95

#The local constraint, the sum of the positive decisions should be less than 50 for each group
local_constraint = {"old": 50, "not_a_real_group": 100, "young": 50} 

#Initialize the solver with the data
my_solver = BinarySolver(y_hat=y_hat, 
                         groups=groups, 
                         cost_matrix=cost_matrix,
                         global_constraint=global_constraint,
                         local_constraint=local_constraint)

#Solve the optimization problem
print(f"Solving now for {N} records")
decisions = my_solver.solve()

results_df = pd.DataFrame({"y_hat": y_hat, "groups": groups, "decisions": decisions})

#Testing:
print("Total number of predicted positives:", decisions.sum())

for group, constraint in local_constraint.items():
    print("Total number of predicted positives for group", group, ":", results_df[results_df["groups"]==group]["decisions"].sum())

Solving now for 9769 records
Total number of predicted positives: 95.0
Total number of predicted positives for group old : 50.0
Total number of predicted positives for group not_a_real_group : 0.0
Total number of predicted positives for group young : 45.0


#### On test set, we can evaluate the performance of the model

In [6]:
def calc_cost(y_true, decisions, cost_matrix):
    individual_tn_cost = (cost_matrix[:, 0, 0] * (1-decisions) * (y_true==0))
    individual_fn_cost = (cost_matrix[:, 1, 0] * (1-decisions) * (y_true==1)) 
    individual_tp_cost = (cost_matrix[:, 1, 1] * decisions * (y_true==1))
    individual_fp_cost = (cost_matrix[:, 0, 1] * decisions * (y_true==0))

    individual_cost = individual_tn_cost + individual_fn_cost + individual_tp_cost + individual_fp_cost
    total_cost = individual_cost.sum()    
    return total_cost

true_cost = calc_cost(y_test, decisions, cost_matrix)
approximated_cost = my_solver.objective_function_value
constraintless_cost = calc_cost(y_test, y_hat>0.75, cost_matrix) #the threshold depends on the cost matrix

In [7]:
print("cost when having constraints:", true_cost)
print("Approximated cost without knowing y_true:", approximated_cost)
print("Constraintless cost:", constraintless_cost)

cost when having constraints: 2199.0
Approximated cost without knowing y_true: 2230.126622932715
Constraintless cost: 1521


#### constraintless cost

In [87]:
print("True cost:", true_cost)
print("Approximated cost:", approximated_cost)
print("Constraintless cost:", constraintless_cost)

True cost: 2199.0
Approximated cost: 2230.126622932715
Constraintless cost: 1510


In [7]:
buga = pd.Series(y_hat, name="y_hat", index = X_test.index)
buga = pd.concat([buga, groups], axis=1)
buga["cost_ratio"] = (cost_matrix[:, 1, 0] * y_hat ) / (cost_matrix[:, 0, 1] * (1-y_hat))
buga = buga.groupby("age").apply(lambda df: df.sort_values("cost_ratio", ascending=False).head(100)["cost_ratio"])
buga = buga.reset_index(level=0)
buga = buga[buga["cost_ratio"]>=1]